In [1]:
# ============================================================
# CEP PROJECT
# Predictive Analysis of Seasonal Diseases in Maharashtra
#
# FILE 5: FEATURE ENGINEERING
# ============================================================

# ------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# ------------------------------------------------------------

from google.colab import drive
drive.mount('/content/drive')


# ------------------------------------------------------------
# 2. IMPORT LIBRARIES
# ------------------------------------------------------------

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 3. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_PATH = "/content/drive/MyDrive/CEP Seasonal Disease Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "Data",
    "processed"
)

INPUT_FILE = os.path.join(
    PROCESSED_PATH,
    "Maharashtra_Seasonal_Disease_Cleaned.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_PATH,
    "Maharashtra_Feature_Engineered.csv"
)

FEATURE_PATH = os.path.join(
    PROCESSED_PATH,
    "Feature_Engineering"
)

os.makedirs(FEATURE_PATH, exist_ok=True)


# ------------------------------------------------------------
# 4. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("Dataset loaded successfully.")
print("Original Shape:", df.shape)

display(df.head())


# ------------------------------------------------------------
# 5. BASIC DATA PREPARATION
# ------------------------------------------------------------

df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

numeric_columns = [
    "Cases",
    "Deaths",
    "preci",
    "LAI",
    "Temp",
    "Latitude",
    "Longitude",
    "week_of_outbreak"
]

for col in numeric_columns:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# Remove rows without valid dates

df = df.dropna(
    subset=["Date"]
).copy()


# ------------------------------------------------------------
# 6. SORT DATA
# ------------------------------------------------------------

sort_columns = [
    col for col in [
        "district",
        "Disease",
        "Date"
    ]
    if col in df.columns
]

df = df.sort_values(
    sort_columns
).reset_index(
    drop=True
)


# ------------------------------------------------------------
# 7. CALENDAR FEATURES
# ------------------------------------------------------------

df["Year"] = df["Date"].dt.year

df["Month"] = df["Date"].dt.month

df["Day"] = df["Date"].dt.day

df["DayOfYear"] = (
    df["Date"].dt.dayofyear
)

df["WeekOfYear"] = (
    df["Date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

df["Quarter"] = (
    df["Date"].dt.quarter
)

df["DaysSinceStart"] = (
    df["Date"] - df["Date"].min()
).dt.days


# ------------------------------------------------------------
# 8. CYCLICAL MONTH FEATURES
# ------------------------------------------------------------

df["Month_sin"] = np.sin(
    2 * np.pi * df["Month"] / 12
)

df["Month_cos"] = np.cos(
    2 * np.pi * df["Month"] / 12
)


# ------------------------------------------------------------
# 9. CYCLICAL WEEK FEATURES
# ------------------------------------------------------------

df["Week_sin"] = np.sin(
    2 * np.pi * df["WeekOfYear"] / 52
)

df["Week_cos"] = np.cos(
    2 * np.pi * df["WeekOfYear"] / 52
)


# ------------------------------------------------------------
# 10. CYCLICAL DAY-OF-YEAR FEATURES
# ------------------------------------------------------------

df["DayOfYear_sin"] = np.sin(
    2 * np.pi * df["DayOfYear"] / 365.25
)

df["DayOfYear_cos"] = np.cos(
    2 * np.pi * df["DayOfYear"] / 365.25
)


# ------------------------------------------------------------
# 11. TEMPERATURE FEATURES
# ------------------------------------------------------------

if "Temp" in df.columns:

    df["Temperature"] = df["Temp"]

    df["Temperature_Squared"] = (
        df["Temp"] ** 2
    )


# ------------------------------------------------------------
# 12. PRECIPITATION FEATURES
# ------------------------------------------------------------

if "preci" in df.columns:

    df["Precipitation_Log"] = np.log1p(
        df["preci"].clip(lower=0)
    )

    df["Rainfall_Present"] = np.where(
        df["preci"].fillna(0) > 0,
        1,
        0
    )


# ------------------------------------------------------------
# 13. LAI FEATURES
# ------------------------------------------------------------

if "LAI" in df.columns:

    df["LAI_Squared"] = (
        df["LAI"] ** 2
    )


# ------------------------------------------------------------
# 14. SEASON ENCODING
# ------------------------------------------------------------

season_mapping = {
    "Summer": 0,
    "Monsoon": 1,
    "Post-Monsoon": 2,
    "Winter": 3
}

if "Season" in df.columns:

    df["Season_Code"] = (
        df["Season"]
        .map(season_mapping)
    )


# ------------------------------------------------------------
# 15. GROUPING FOR HISTORICAL FEATURES
# ------------------------------------------------------------

group_columns = [
    "district",
    "Disease"
]

if all(
    col in df.columns
    for col in group_columns
):

    # --------------------------------------------------------
    # Previous Cases
    # --------------------------------------------------------

    df["Previous_Cases"] = (
        df.groupby(
            group_columns
        )["Cases"]
        .shift(1)
    )


    # --------------------------------------------------------
    # Lag 2
    # --------------------------------------------------------

    df["Cases_Lag_2"] = (
        df.groupby(
            group_columns
        )["Cases"]
        .shift(2)
    )


    # --------------------------------------------------------
    # Lag 3
    # --------------------------------------------------------

    df["Cases_Lag_3"] = (
        df.groupby(
            group_columns
        )["Cases"]
        .shift(3)
    )


    # --------------------------------------------------------
    # Rolling 3
    #
    # IMPORTANT:
    # shift(1) means current Cases is NOT included.
    # --------------------------------------------------------

    df["Cases_Rolling_3"] = (
        df.groupby(
            group_columns
        )["Cases"]
        .transform(
            lambda x:
            x.shift(1)
             .rolling(
                 window=3,
                 min_periods=1
             )
             .mean()
        )
    )


    # --------------------------------------------------------
    # Rolling 6
    # --------------------------------------------------------

    df["Cases_Rolling_6"] = (
        df.groupby(
            group_columns
        )["Cases"]
        .transform(
            lambda x:
            x.shift(1)
             .rolling(
                 window=6,
                 min_periods=1
             )
             .mean()
        )
    )


# ------------------------------------------------------------
# 16. CASE TREND
# ------------------------------------------------------------

if (
    "Previous_Cases" in df.columns
    and "Cases_Lag_2" in df.columns
):

    df["Case_Change"] = (
        df["Previous_Cases"]
        - df["Cases_Lag_2"]
    )


# ------------------------------------------------------------
# 17. CASE GROWTH RATE
# ------------------------------------------------------------

if (
    "Previous_Cases" in df.columns
    and "Cases_Lag_2" in df.columns
):

    denominator = (
        df["Cases_Lag_2"]
        .replace(0, np.nan)
    )

    df["Case_Growth_Rate"] = (
        (
            df["Previous_Cases"]
            - df["Cases_Lag_2"]
        )
        / denominator
    )


# ------------------------------------------------------------
# 18. PREVIOUS DEATHS
# ------------------------------------------------------------

if "Deaths" in df.columns:

    df["Previous_Deaths"] = (
        df.groupby(
            group_columns
        )["Deaths"]
        .shift(1)
    )


# ------------------------------------------------------------
# 19. PREVIOUS CLIMATE FEATURES
# ------------------------------------------------------------

if "Temp" in df.columns:

    df["Previous_Temperature"] = (
        df.groupby(
            group_columns
        )["Temp"]
        .shift(1)
    )


if "preci" in df.columns:

    df["Previous_Precipitation"] = (
        df.groupby(
            group_columns
        )["preci"]
        .shift(1)
    )


if "LAI" in df.columns:

    df["Previous_LAI"] = (
        df.groupby(
            group_columns
        )["LAI"]
        .shift(1)
    )


# ------------------------------------------------------------
# 20. DISTRICT-LEVEL HISTORICAL CASES
# ------------------------------------------------------------

if (
    "district" in df.columns
    and "Cases" in df.columns
):

    district_cases = (
        df.groupby("district")["Cases"]
    )

    df["District_Previous_Cases"] = (
        district_cases.shift(1)
    )

    df["District_Cases_Rolling_3"] = (
        district_cases.transform(
            lambda x:
            x.shift(1)
             .rolling(
                 window=3,
                 min_periods=1
             )
             .mean()
        )
    )


# ------------------------------------------------------------
# 21. DISEASE-LEVEL HISTORICAL CASES
# ------------------------------------------------------------

if (
    "Disease" in df.columns
    and "Cases" in df.columns
):

    disease_cases = (
        df.groupby("Disease")["Cases"]
    )

    df["Disease_Previous_Cases"] = (
        disease_cases.shift(1)
    )


# ------------------------------------------------------------
# 22. MISSING VALUE INDICATORS
# ------------------------------------------------------------

missing_feature_columns = [
    "Temp",
    "preci",
    "LAI",
    "Deaths",
    "Previous_Cases",
    "Cases_Lag_2",
    "Cases_Lag_3",
    "Cases_Rolling_3",
    "Cases_Rolling_6"
]

for col in missing_feature_columns:

    if col in df.columns:

        df[f"{col}_Missing"] = (
            df[col]
            .isna()
            .astype(int)
        )


# ------------------------------------------------------------
# 23. ONE-HOT ENCODE SEASON
# ------------------------------------------------------------

if "Season" in df.columns:

    season_dummies = pd.get_dummies(
        df["Season"],
        prefix="Season",
        dtype=int
    )

    df = pd.concat(
        [
            df,
            season_dummies
        ],
        axis=1
    )


# ------------------------------------------------------------
# 24. ONE-HOT ENCODE DISEASE
# ------------------------------------------------------------

if "Disease" in df.columns:

    disease_dummies = pd.get_dummies(
        df["Disease"],
        prefix="Disease",
        dtype=int
    )

    df = pd.concat(
        [
            df,
            disease_dummies
        ],
        axis=1
    )


# ------------------------------------------------------------
# 25. ONE-HOT ENCODE DISTRICT
# ------------------------------------------------------------

if "district" in df.columns:

    district_dummies = pd.get_dummies(
        df["district"],
        prefix="District",
        dtype=int
    )

    df = pd.concat(
        [
            df,
            district_dummies
        ],
        axis=1
    )


# ------------------------------------------------------------
# 26. CLEAN INFINITE VALUES
# ------------------------------------------------------------

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)


# ------------------------------------------------------------
# 27. FEATURE INFORMATION
# ------------------------------------------------------------

feature_information = pd.DataFrame({
    "Feature": df.columns,
    "Data_Type": df.dtypes.astype(str),
    "Missing_Values": df.isna().sum().values,
    "Missing_Percentage": (
        df.isna().mean().values * 100
    )
})

feature_information[
    "Missing_Percentage"
] = feature_information[
    "Missing_Percentage"
].round(2)

feature_information.to_csv(
    os.path.join(
        FEATURE_PATH,
        "feature_information.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 28. FEATURE ENGINEERING SUMMARY
# ------------------------------------------------------------

original_df = pd.read_csv(
    INPUT_FILE
)

feature_summary = pd.DataFrame({
    "Metric": [
        "Original Rows",
        "Final Rows",
        "Original Columns",
        "Final Columns",
        "New Features"
    ],
    "Value": [
        len(original_df),
        len(df),
        len(original_df.columns),
        len(df.columns),
        len(df.columns)
        - len(original_df.columns)
    ]
})

print("\nFeature Engineering Summary:")

display(feature_summary)

feature_summary.to_csv(
    os.path.join(
        FEATURE_PATH,
        "feature_engineering_summary.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 29. IMPORTANT FEATURES CHECK
# ------------------------------------------------------------

important_features = [
    "Date",
    "Year",
    "Month",
    "WeekOfYear",
    "Season",
    "Season_Code",
    "Month_sin",
    "Month_cos",
    "Week_sin",
    "Week_cos",
    "Temperature",
    "Temperature_Squared",
    "Precipitation_Log",
    "Rainfall_Present",
    "Previous_Cases",
    "Cases_Lag_2",
    "Cases_Lag_3",
    "Cases_Rolling_3",
    "Cases_Rolling_6",
    "Case_Change",
    "Case_Growth_Rate",
    "Previous_Deaths",
    "Previous_Temperature",
    "Previous_Precipitation",
    "Previous_LAI",
    "District_Previous_Cases",
    "District_Cases_Rolling_3",
    "Disease_Previous_Cases"
]

available_features = [
    col for col in important_features
    if col in df.columns
]

print("\nImportant Engineered Features:")

display(
    df[
        available_features
    ].head(15)
)


# ------------------------------------------------------------
# 30. SAVE FEATURE-ENGINEERED DATASET
# ------------------------------------------------------------

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nFeature-engineered dataset saved successfully.")


# ------------------------------------------------------------
# 31. FEATURE CORRELATION
# ------------------------------------------------------------

feature_numeric = df.select_dtypes(
    include=np.number
)

if (
    "Cases" in feature_numeric.columns
    and len(feature_numeric.columns) > 1
):

    feature_correlation = (
        feature_numeric.corr()
    )

    feature_correlation.to_csv(
        os.path.join(
            FEATURE_PATH,
            "feature_correlation_matrix.csv"
        )
    )

    cases_correlation = (
        feature_correlation["Cases"]
        .sort_values(
            ascending=False
        )
        .to_frame(
            name="Correlation_with_Cases"
        )
    )

    print(
        "\nTop Features Correlated With Cases:"
    )

    display(
        cases_correlation.head(20)
    )

    cases_correlation.to_csv(
        os.path.join(
            FEATURE_PATH,
            "feature_correlation_with_cases.csv"
        )
    )


# ------------------------------------------------------------
# 32. DUPLICATE CHECK
# ------------------------------------------------------------

duplicate_count = (
    df.duplicated().sum()
)

print(
    "\nDuplicate rows:",
    duplicate_count
)


# ------------------------------------------------------------
# 33. FINAL DATASET INFORMATION
# ------------------------------------------------------------

print("\n============================================================")
print("FINAL FEATURE-ENGINEERED DATASET")
print("============================================================")

print(
    "\nFinal Shape:",
    df.shape
)

print(
    "\nTotal Features:",
    len(df.columns)
)

print("\nTop Missing Values:")

display(
    df.isna()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(20)
)


# ------------------------------------------------------------
# 34. LIST OUTPUT FILES
# ------------------------------------------------------------

print("\n============================================================")
print("FEATURE ENGINEERING COMPLETED SUCCESSFULLY")
print("============================================================")

print("\nMain Dataset:")
print(OUTPUT_FILE)

print("\nSupporting Files:")

for file_name in sorted(
    os.listdir(FEATURE_PATH)
):

    print("✓", file_name)

print("\nFeature Engineering Folder:")
print(FEATURE_PATH)

Mounted at /content/drive
Dataset loaded successfully.
Original Shape: (1183, 17)


,Date,year,mon,day,week_of_outbreak,Season,state_ut,district,Disease,Cases,Deaths,Deaths_missing,Latitude,Longitude,preci,Temp,LAI
0,2012-06-22,2012,6,22,NaN,Monsoon,Maharashtra,Ahmednagar,Acute Diarrhoeal Disease,15,NaN,1,19.162772,74.858024,0.333666,310.492000,3.0
1,2012-06-29,2012,6,29,NaN,Monsoon,Maharashtra,Ahmednagar,Acute Diarrhoeal Disease,17,NaN,1,19.162772,74.858024,0.051972,312.355000,3.0
2,2012-10-09,2012,10,9,NaN,Post-Monsoon,Maharashtra,Ahmednagar,Acute Diarrhoeal Disease,93,NaN,1,19.162772,74.858024,1.188847,302.340000,11.0
3,2013-09-25,2013,9,25,NaN,Monsoon,Maharashtra,Ahmednagar,Acute Diarrhoeal Disease,20,NaN,1,19.162772,74.858024,0.454465,301.566667,13.0
4,2015-05-27,2015,5,27,NaN,Summer,Maharashtra,Ahmednagar,Acute Diarrhoeal Disease,15,NaN,1,19.162772,74.858024,0.000634,317.870000,3.0



Feature Engineering Summary:


,Metric,Value
0,Original Rows,1183
1,Final Rows,1183
2,Original Columns,17
3,Final Columns,109
4,New Features,92



Important Engineered Features:


...


Feature-engineered dataset saved successfully.

Top Features Correlated With Cases:


,Correlation_with_Cases
Cases,1.000000
District_Aurangabad,0.206954
Disease_Cholera,0.163717
Cases_Lag_2,0.141459
Cases_Lag_3,0.107273
Deaths,0.103825
Cases_Lag_3_Missing,0.058716
Cases_Lag_2_Missing,0.051412
Season_Post-Monsoon,0.045505
Disease_Acute Diarrhoeal Disease,0.035605



Duplicate rows: 0

FINAL FEATURE-ENGINEERED DATASET

Final Shape: (1183, 109)

Total Features: 109

Top Missing Values:


,0
week_of_outbreak,1183
Previous_Deaths,892
Deaths,842
Previous_LAI,437
Cases_Lag_3,386
LAI_Squared,320
LAI,320
Cases_Lag_2,291
Case_Growth_Rate,291
Case_Change,291



FEATURE ENGINEERING COMPLETED SUCCESSFULLY

Main Dataset:
/content/drive/MyDrive/CEP Seasonal Disease Prediction/Data/processed/Maharashtra_Feature_Engineered.csv

Supporting Files:
✓ feature_correlation_matrix.csv
✓ feature_correlation_with_cases.csv
✓ feature_engineering_summary.csv
✓ feature_information.csv

Feature Engineering Folder:
/content/drive/MyDrive/CEP Seasonal Disease Prediction/Data/processed/Feature_Engineering


In [2]:
# ============================================================
# SAVE FEATURE-ENGINEERED DATA
# ============================================================

FEATURE_PATH = os.path.join(
    PROJECT_PATH,
    "Data",
    "processed",
    "Feature_Engineering"
)

os.makedirs(FEATURE_PATH, exist_ok=True)

OUTPUT_FILE = os.path.join(
    FEATURE_PATH,
    "Maharashtra_Feature_Engineered.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("FEATURE ENGINEERING DATA SAVED")
print("=" * 70)

print("\nFile:")
print(OUTPUT_FILE)

print("\nShape:")
print(df.shape)

print("\nFile Exists:")
print(os.path.exists(OUTPUT_FILE))


FEATURE ENGINEERING DATA SAVED

File:
/content/drive/MyDrive/CEP Seasonal Disease Prediction/Data/processed/Feature_Engineering/Maharashtra_Feature_Engineered.csv

Shape:
(1183, 109)

File Exists:
True
